In [ ]:

#importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# First Model:
## Step 1: Loading the Data
We import our cleaned_data with all data needed

In [ ]:
df  = pd.read_csv('cleaned_dataset.csv', sep=';')

## Step 2: Defining the Goal (Target) and what is usable (Features)
To train an AI, we must tell it what it needs to guess (the **Target**) and what information it is allowed to use (the **Features**). 

Then, we clean the table to remove rows where the delay is missing.

In [ ]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season']

# We remove rows where the delay is equal to -1 and nan (meaning the data was invalid during cleaning).
df = df[df[target] != -1]
df = df.dropna()

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 3: Translating Text for the Computer (Encoding)
An Artificial Intelligence is a calculating machine: it only understands mathematics. It cannot read words like "Paris" or "Bordeaux". 
We must therefore use a "translator" to convert these station names into numerical codes that the computer can analyze. Encoding of columns where the value isn't usable by the model (strings) into and usable value

In [ ]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

## Step 4: Creating the AI "Assembly Line"
We are going to set up our prediction model. The chosen algorithm is called a **Random Forest**. 
It combines the output of multiple decision trees to reach a single result

In [ ]:
# We create a "Pipeline": it is an automated assembly line.
model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

## Step 5: Creating the final AI
This is the most important step. We will divide our data into two batches:
- **80% for training:** The AI practices guessing the delays and looks at the real answers to learn from its mistakes.
- **20% for testing:** We hide the answers from the AI and ask it to make its predictions to see if it has understood the logic.

In [ ]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred = model.predict(X_test)

## Step 6: Grading (Performance Evaluation)
Now that the AI has taken its exam and made its predictions, we will compare its answers with reality to give it performance grades.

In [ ]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : {mean_absolute_error(y_test, y_pred):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : {mean_squared_error(y_test, y_pred):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : {r2_score(y_test, y_pred):.2f}")

## Conclusion on the first model
For a first model, it's not bad but it's not good either. A R² score of 0.25 mean that the model is slightly better than pur randomness.
This mean that this model can be tuned even more to try to have a better score and so a better prediction model.

# Second Model
We saw previously that our model had a score of 0.25, but we need to take in mind that it only use direct or easily deductible parameters. But with what we have now, maybe we could deduce new parameters to give him. For example, if we know the departure and arrival stations, we can do a mean of scheduled train for this trip during this season. It is not the most accurate parameter but it is still better than we don t give any parameter at all.

## Step 1: Adding new parameters
We can add new parameters like : Number of scheduled trains, Number of cancelled trains, Number of trains delayed at departure, Number of delayed trains at arrival

In [ ]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season', 'Number of scheduled trains', 'Number of trains delayed > 15min']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 2 : Encoding

In [ ]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

## Step 3: Creating the AI "Assembly Line"

In [ ]:
# We create a "Pipeline": it is an automated assembly line.
model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

## Step 4: Creating the final AI

In [ ]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred = model.predict(X_test)

## Step 5: Grading (Performance Evaluation)

In [ ]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : {mean_absolute_error(y_test, y_pred):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : {mean_squared_error(y_test, y_pred):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : {r2_score(y_test, y_pred):.2f}")

## Conclusion
We saw that adding new parameters don t necessarly increase the efficiency of the model so we need to think of others things

# Hyperparameters
One solution to improve the model could be the Hyperparamters. Hyperparameters are parameters the guide how the model must learn. In our case we will use Grid Search. Grid Search is an algorythm that will try every combination of hyperparameters to search the best one.

## Step 1 : Setting up the data
We set up everything the same way that before

In [ ]:
target = 'Average delay of all trains at arrival'
#We go back to previous features because we saw that they didn't improve the model
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

# We create a "Pipeline": it is an automated assembly line.
model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Step 2 : We setup the grid search to try differents parameters
We need to initiate which parameters the grid search will use and tru to combine

In [ ]:
# We split the cols into categorical and numeric columns
categorical_cols = ['Service', 'Departure station', 'Arrival station', 'Season']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# We transform the categoricals values into numerical values
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# We create a pipeline that first transforms the data and then applies the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])


# We define the grid of hyperparameters to search over
param_grid = {
    # We search for the best number of trees in the random forest
    'model__n_estimators': [50, 200, 500],
    # We search for the best maximum depth of the trees in the random forest
    'model__max_depth': [None, 20, 50],
    # We search for the best minimum number of samples required to split an internal node
    'model__min_samples_split': [2, 10, 20],
    # We search for the best minimum number of samples required to be at a leaf node
    'model__min_samples_leaf': [1, 6, 10]
}

# We create the Gridsearch algorythm that will search for the best hyperparameters
grid_search = GridSearchCV(
    # The model to tune and evaluate
    estimator=pipeline, 
    # The dictionary containing the hyperparameters to test
    param_grid=param_grid,
    # Number of cross-validation folds (splits data into 5 parts)
    cv=5,
    # The metric used to evaluate and compare the models' performance
    scoring='r2',
    # Uses all available CPU cores to run calculations in parallel
    n_jobs=-1
)

# We split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 3 : Executing the grid search

Now that everything is setup, we can execute it and see print the results of the grid search 

In [ ]:
# We fit the grid search to the training data
grid_search.fit(X_train, y_train)

# We print the best hyperparameters found by the grid search
print("Best hyperparameters :", grid_search.best_params_)

The grid search algorythm found that the best parameters are depth of 30, leaf of 1, split of 10, ans estimators of 400.
We can now create a new model where we apply the hyperparameters and see if it's better.

# Model 3

## Step 1 : initialization of the model
First we init the model like we did before


In [ ]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

# We create a "Pipeline": it is an automated assembly line.
model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

## Step 2 : Apply hyperparameters
Now that we did the setup we can apply the hyperparameters we found to the model

In [ ]:
# Apply hyperparameter we found with GridSearchCV
model.set_params(regressor__n_estimators=400)
model.set_params(regressor__max_depth=30)
model.set_params(regressor__min_samples_split=10)
model.set_params(regressor__min_samples_leaf=1)

## Step 3 : Finishing the Ai
We can finally finish the Ai and test it after

In [ ]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred = model.predict(X_test)

## Step 4 : Grading the Ai
We can now grade the model to see if it improved

In [ ]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : {mean_absolute_error(y_test, y_pred):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : {mean_squared_error(y_test, y_pred):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : {r2_score(y_test, y_pred):.2f}")